# Risk Analytics of a Portfolio

## 1. Portfolio Return and Volatility

Let a portfolio contain $n$ assets and let $R_i$ denote the random **rate of return** of the $i$-th asset, that is, $R_i$ is a random variable. Let $\mu_i = E[R_i]$ be the **expected rate of return** of the $i$-th asset. Suppose $w_i$ denotes the weight of the $i$-th asset in the portfolio, and so $\sum_{i = 1}^n w_i = 1$.

The **rate of return of the portolio** is given by:
$$ R_p = w_1 R_1 + w_2 R_2 + ... + w_n R_n = \sum_{i = 1}^n w_i R_i $$
and so, by the linearity of expectation, 
$$ E[R_p] = \sum_{i = 1}^n w_i E[R_i]. $$

If $\mu_p$ denotes the rate of return of the portfolio, 
$$\boxed{\mu_p = \sum_{i = 1}^n w_i \mu_i \tag{1}} $$

Let $\sigma_p$ denote the **volatility** of the portfolio as the **standard deviation** of the rate of return of the portfolio, i.e., $\sigma_p = \sqrt{Var(R_p)}$. 

\begin{align*}
Var(R_p) &= E[(R_p - E[R_p])^2] \\
&= E\left[\left( \sum_{i = 1}^n w_i R_i - \sum_{i = 1}^n w_i E[R_i] \right)^2\right] \\
&= E\left[\left( \sum_{i = 1}^n w_i (R_i - E[R_i]) \right)^2\right] \\
&= E\left[\left( \sum_{i = 1}^n w_i (R_i - E[R_i]) \right)\left( \sum_{j = 1}^n w_j (R_j - E[R_j]) \right) \right] \\
&= E \left[ \sum_{i, j = 1}^n w_i w_j (R_i - E[R_i]) (R_j - E[R_j]) \right] \\
&= \sum_{i, j = 1}^n w_i w_j E[(R_i - E[R_i]) (R_j - E[R_j])] \\ 
&= \sum_{i, j = 1}^n w_i w_j \sigma_{ij}
\end{align*}

where $\sigma_{ij}$ is $Cov(R_i, R_j)$, the covariance of the rates of return of the $i$-th and $j$-th asset. Therefore, 

$$ \boxed{\sigma_p = \sqrt{\sum_{i, j = 1}^n w_i w_j \sigma_{ij}}} \tag{2}$$

### 1.1 Matrix representation

Let 
$$
\boldsymbol{\mu} = 
\begin{pmatrix}
\mu_1 \\
\mu_2 \\
\vdots \\
\mu_n
\end{pmatrix}, \quad
\boldsymbol{w} =
\begin{pmatrix}
w_1 \\
w_2 \\
\vdots \\
w_n
\end{pmatrix}, \quad
\boldsymbol{\Sigma} = 
\begin{pmatrix}
\sigma_1^2 & \sigma_{12} & \cdots & \sigma_{1n} \\
\sigma_{21} & \sigma_2^2 & \cdots & \sigma_{2n} \\
\vdots & \vdots & \ddots & \vdots \\
\sigma_{n1} & \sigma_{n2} & \cdots & \sigma_n^2.
\end{pmatrix}
$$

Then, $(1)$ can be written as:

$$\boxed{\mu_p = \boldsymbol{w}^\top \boldsymbol{\mu}} \tag{3}$$

and $(2)$ can be written as:

$$\boxed{\sigma_p = \sqrt{\boldsymbol{w}^\top \boldsymbol{\Sigma} \boldsymbol{w}}} \tag{4}$$



### 1.2 Code

In [1]:
import numpy as np

In [2]:
"""
Calculates the rate of return and the volatility of the portfolio.

mu: ndarray (n,) containing the rates of return of n assets in the portfolio as decimals (e.g., 23% is written as 0.23)
w: ndarray (n,) containing the weights of n assets in the portfolio summing to 1
Sigma: ndarray (n, n), covariance matrix

Returns: a dictionary containing the rate of return and volatility of the portfolio
"""

def ret_and_vol(
    mu: np.ndarray, 
    w: np.ndarray, 
    Sigma: np.ndarray
) -> dict[str, float]:
    
    ret = w @ mu
    vol = np.sqrt(w @ Sigma @ w)
    
    return {
        "Return" : ret,
        "Volatility" : vol
    } 

## 2. Historical VaR

Consider a portfolio with $n$ assets. Let $P_{i, T}$ denote the price of the asset $i$ at time $T$, where $T = 0, 1, ..., m$. The **rate of return** of the $i$-th asset between time $t$ and $t + 1$ is given by
$$ R_{i, t} = \frac{P_{i, t + 1} - P_{i, t}}{P_{i, t}} $$
where $t = 0, 1, ..., m - 1$. 

If $w_i$ denotes the weight of the $i$-th asset in the portfolio, the rate of return of the portfolio between time $t$ and $t + 1$ is
$$ R_{p, t} = \sum_{i = 1}^n w_i R_{i, t}. $$

For example, suppose we have $5$ assets and we have the historical prices of these $5$ assets over 1000 days (here, $m = 999$). We can then calculate the rate of return for each of the $999$ time periods for each asset, and by using the weights of the assets, we can calculate the rate of return of the portfolio for each time period. We can then **sort** the rates of return thus obtained from worst to best.

Suppose that our **loss tolerance** is $5\%$ (i.e., **confidence level** is $95\%$). Then, we take the $5$th percentile of the return distribution, say $-0.03$: a **one-day loss** exceeding $3\%$ of the portfolio value occurred in about $5\%$ of the historical observations. If the current portfolio value is $\$100,000$, then we say that the one-day **95% historical VaR (value at risk)** is $\$3000$.

Here, we make no assumption that the returns are normally distributed and simply use the historical data.

### 2.1 Historical CVaR

Let the loss tolerance be $L\%$. The **Conditional Value at Risk** or **Expected Shortfall** attemps to answer the question: what is the average loss below the $L$th percentile?

For example, if the loss tolerance is $5\%$ and the $5$th percentile of the return distribution is $-0.03$, then to find the historical CVaR, we take the mean of all the one-day returns below $-0.03$. If that mean is $-0.07$ and the portfolio value is $\$100,000$, then the expected shortfall is $\$7,000$.

### 2.2 Code

In [3]:
"""
Calculates the (daily) historical VaR and CVaR of the portfolio.

asset_prices: ndarray (t x n), with rows representing days and columns representing assets
weights: ndarray (n,) containing the weights of n assets in the portfolio summing to 1
total_amount: current portfolio value
confidence: confidence level in percentage, e.g., 95

Returns: a list containing:
    the rate of return at the specified loss tolerance percentile
    the value at risk
    expected shortfall rate of return for the specified confidence interval
    expected shortfall
"""

def hist_var_cvar(
    asset_prices: np.ndarray, 
    weights: np.ndarray, 
    total_amount: float, 
    confidence: float
) -> list[float]:
    
    asset_returns = (asset_prices[1:] - asset_prices[:-1]) / asset_prices[:-1]
    portfolio_returns = asset_returns @ weights
    
    var_return = np.percentile(portfolio_returns, 100 - confidence)
    var = var_return * total_amount

    tail_returns = portfolio_returns[portfolio_returns <= var_return]
    cvar_return = np.mean(tail_returns)
    cvar = cvar_return * total_amount
    
    return [-var_return, -var, -cvar_return, -cvar]

## 3. Parametric VaR

Let $\mu_p$ be the portfolio return and $\sigma_p$ be the portfolio volatility. We assume that the returns on the portfolio, $R_p$, are normally distributed. Then, 

$$ R_p \sim \mathcal{N}(\mu_p, \sigma_p^2). $$

Now, if our confidence level is $95\%$ and the $5$th percentile of the distribution of $R_p$ is $-0.04$, then the VaR is $0.04P$, where $P$ is the portfolio value. 

$R_p$ can also be expressed as:
$$ R_p = \mu_p + \sigma_p Z $$
where $Z \sim \mathcal{N}(0, 1)$, and so 

$$ VaR_{0.95} = P \cdot (\mu_p + \sigma_p\Phi^{-1}(0.05)) $$
where $\Phi$ is the CDF of $Z$.

### 3.1 Parametric CVaR

Again, the CVaR is the expected loss given that the loss exceeds VaR:

$$ \boxed{CVaR_{c} = E[L | L \geq VaR_{c}]} $$

where $L$ is the loss on the portfolio. 

Now, let $\alpha$ be the loss tolerance in decimal (not percentage), e.g., 0.05. Let $q_{\alpha} = \mu_p + \sigma_p \Phi^{-1}(\alpha)$, the corresponding percentile of the distribution of $R_p$.

$$ E[R_p | R_p \leq q_{\alpha}] =  \frac{1}{P(R_p \leq q_{\alpha})} \int_{-\infty}^{q_\alpha} xf_{R_p}(x)dx = \frac{1}{\alpha} \int_{-\infty}^{q_\alpha} xf_{R_p}(x)dx$$

and by changing the variable $x = \mu_p + \sigma_p z$, we obtain

$$ \boxed{E[R_p | R_p \leq q_\alpha] = \mu_p - \sigma_p \frac{\phi(z)}{\alpha}}$$

where $z = \frac{q_\alpha - \mu_p}{\sigma_p}$.

### 3.2 Code

In [4]:
from scipy.stats import norm

In [5]:
"""
Calculates the parametric VaR and CVaR of the portfolio,
assuming portfolio returns are normally distributed.

mu: ndarray (n,), containing the expected returns of n assets
w: ndarray (n,), containing the weights of n assets in the portfolio
Sigma: ndarray (n x n), covariance matrix of the asset returns
total_amount: current portfolio value
confidence: confidence level in percentage, e.g., 95

Returns: a list containing:
    the rate of return at the specified loss tolerance percentile
    the value at risk
    expected shortfall rate of return for the specified confidence interval
    expected shortfall
"""

def parametric_var_cvar(
    mu: np.ndarray, 
    w: np.ndarray, 
    Sigma: np.ndarray, 
    total_amount: float, 
    confidence: float
) -> list[float]:
    
    mu_p, sigma_p = ret_and_vol(mu, w, Sigma)
    alpha = (100 - confidence) / 100
    
    var_return = mu_p + sigma_p * norm.ppf(alpha)
    var = var_return * total_amount

    z = (var_return - mu_p) / sigma_p
    cvar_return = mu_p - sigma_p * norm.pdf(z) / alpha
    cvar = cvar_return * total_amount

    return [-var_return, -var, -cvar_return, -cvar]

## 4. Maximum Drawdown

The **maximum drawdown (MDD)** is the largest peak-to-trough decline in portfolio value. 

Let the value of the portfolio be $V_0, V_1, ..., V_T$, where $V_t$ denotes its value at time $t$. Define the running maximum:
$$ M_t := \max_{0 \leq s \leq t} V_s. $$

The **drawdown** at time $t$ is
$$ D_t = \frac{V_t - M_t}{M_t}. $$

Since $V_t \leq M_t$, $D_t \leq 0$. The **maximum drawdown** is given 
$$ \boxed{MDD = \min_{t} D_t}. $$

### 4.1 Code

In [6]:
"""
Calculates the maximum drawdown of a portfolio over the given
period.

asset_prices: ndarray (T x n), containing the prices of n assets
    over T time periods
weights: ndarray (n,), containing the weights of n assets in the
    portfolio

Returns: the maximum drawdown of the portfolio as a decimal.
    The value is negative, e.g. -0.20 represents a 20% maximum
    drawdown.
"""

def mdd(
    asset_prices: np.ndarray, 
    weights: np.ndarray
) -> float:
    
    v = (asset_prices / asset_prices[0]) @ weights
    peak = -np.inf
    max_dd = 0
    
    for value in v:
        peak = max(peak, value)
        drawdown = (value - peak) / peak
        max_dd = min(max_dd, drawdown)
    
    return max_dd

## 5. Sharpe Ratio

The **Sharpe ratio** is defined as

$$ \boxed{S = \frac{E[R_p] - r}{\sigma_p}}. $$



### 5.1 Code

In [7]:
"""
Calculates the annualized Sharpe ratio of a portfolio over the
given period.

asset_prices: ndarray (T x n), containing the prices of n assets
    over T time periods
weights: ndarray (n,), containing the weights of n assets in the
    portfolio
r: annualized risk-free rate as a decimal

Returns: the annualized Sharpe ratio of the portfolio.
"""

def sharpe(
    asset_prices: np.ndarray, 
    weights: np.ndarray,
    r: float
) -> float:
    
    asset_returns = (asset_prices[1:] - asset_prices[:-1]) / asset_prices[:-1]
    mu = np.mean(asset_returns, axis=0)
    Sigma = np.cov(asset_returns.T)

    mu_p, sigma_p = ret_and_vol(mu, weights, Sigma)
    
    S = np.sqrt(252) * (mu_p - r / 252) / sigma_p
    
    return S